In [ ]:
# Cell 1: Fine-tune MiniLM on Fault Descriptions and extract fine-tuned embeddings

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from tqdm import tqdm

# Load data
# df = pd.read_excel("fault-logs-o-ran-12k.xlsx", engine="openpyxl")
df = pd.read_excel("fault-logs-o-ran-64.xls")

fault_texts = df["Fault Description"].fillna("").astype(str).tolist()


# Load MiniLM tokenizer and model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Custom dataset
class FaultDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(self.texts[idx], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
        return {key: val.squeeze(0) for key, val in encoded.items()}

# DataLoader
dataset = FaultDataset(fault_texts)
loader = DataLoader(dataset, batch_size=16, shuffle=False)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Fine-tune MiniLM (light fine-tuning)
model.train()
for epoch in range(1):  # Keep it low to avoid overfitting
    for batch in tqdm(loader, desc="Fine-tuning MiniLM"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        # Self-supervised dummy objective (contrastive not implemented here for simplicity)
        loss = sentence_embeddings.norm(p=2, dim=1).mean()  # L2 regularization proxy
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

# Re-extract embeddings using fine-tuned MiniLM
model.eval()
all_embeddings = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting Fine-tuned Embeddings"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(embeddings.cpu())

final_embeddings = torch.cat(all_embeddings, dim=0)
print(f"Final Embedding Shape: {final_embeddings.shape}")  # Shape: [num_samples, hidden_dim]


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Number of generators
num_generators = 30

# Generator
class Generator(nn.Module):
    def __init__(self, latent_dim=100, output_dim=384):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 256),
            nn.ReLU(True),
            nn.Linear(256, 256),
            nn.ReLU(True),
            nn.Linear(256, output_dim),
        )

    def forward(self, z):
        return self.model(z)

# Create generators
generators = [
    Generator(latent_dim=100, output_dim=384).to(device)
    for _ in range(num_generators)
]

# MAD-GAN Discriminator
class MADDiscriminator(nn.Module):
    def __init__(self, input_dim=384, num_generators=30):
        super(MADDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, num_generators + 1),
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        logits = self.net(x)
        return self.softmax(logits)

# Create discriminator
discriminator = MADDiscriminator(
    input_dim=384,
    num_generators=num_generators
).to(device)

In [ ]:
import torch.nn.functional as F

# Hyperparameters
latent_dim = 100
hidden_dim = final_embeddings.shape[1]  # 384
batch_size = 64
num_epochs = 500
lr = 1e-4
K = 30  # Number of generators

# Create multiple generators
generators = [Generator(latent_dim, hidden_dim).to(device) for _ in range(K)]
D = MADDiscriminator(input_dim=hidden_dim, num_generators=K).to(device)

# Optimizers
opt_G = [torch.optim.Adam(G.parameters(), lr=lr) for G in generators]
opt_D = torch.optim.Adam(D.parameters(), lr=lr)

# Loss function
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

# Dataset and loader
real_dataset = torch.utils.data.TensorDataset(final_embeddings)
real_loader = torch.utils.data.DataLoader(real_dataset, batch_size=batch_size, shuffle=True)

# Training loop
for epoch in range(num_epochs):
    for x_batch, in real_loader:
        x_batch = x_batch.to(device)
        bsz = x_batch.size(0)

        # ============================
        # 1. Train Discriminator
        # ============================

        # Real data → label 0
        real_labels = torch.zeros(bsz, dtype=torch.long).to(device)  # class 0 = real
        real_preds = D(x_batch)  # shape: [bsz, K+1]
        loss_real = ce_loss(real_preds, real_labels)

        # Fake data from all generators → label j ∈ [1..K]
        fake_data = []
        fake_labels = []
        for j, G in enumerate(generators):
            z = torch.randn(bsz, latent_dim).to(device)
            x_fake = G(z).detach()
            fake_preds = D(x_fake)  # shape: [bsz, K+1]
            labels = torch.full((bsz,), j + 1, dtype=torch.long).to(device)  # class j+1 = G_j
            loss_fake = ce_loss(fake_preds, labels)
            fake_data.append((x_fake, labels, loss_fake))

        # Combine all losses
        loss_D = loss_real + sum(lf for _, _, lf in fake_data)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ============================
        # 2. Train each Generator
        # ============================

        loss_G_total = 0.0
        loss_MSE_total = 0.0

        for j, G in enumerate(generators):
            z = torch.randn(bsz, latent_dim).to(device)
            x_fake = G(z)
            preds = D(x_fake)
            target_class = torch.zeros(bsz, dtype=torch.long).to(device)  # Wants to be seen as "real" (class 0)

            loss_G = ce_loss(preds, target_class)

            # Optional MSE term (to guide towards data manifold)
            idx = torch.randint(0, final_embeddings.size(0), (bsz,))
            x_real_ref = final_embeddings[idx].to(device)
            loss_MSE = mse_loss(x_fake, x_real_ref)

            total_loss = loss_G + loss_MSE
            opt_G[j].zero_grad()
            total_loss.backward()
            opt_G[j].step()

            loss_G_total += loss_G.item()
            loss_MSE_total += loss_MSE.item()

    print(f"Epoch {epoch+1} | D Loss: {loss_D.item():.4f} | Avg G Loss: {loss_G_total/K:.4f} | Avg MSE: {loss_MSE_total/K:.4f}")

# Save all generators and discriminator
for j, G in enumerate(generators):
    torch.save(G.state_dict(), f"trained_generator_{j+1}.pt")

torch.save(D.state_dict(), "trained_discriminator.pt")




In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
from tqdm import tqdm

# Load fault texts
df = pd.read_excel("fault-logs-o-ran-350.xls")
#df = pd.read_excel("proj-training-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Load MiniLM tokenizer + model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

max_length = 128
token_level_embeddings = []

with torch.no_grad():
    for sentence in tqdm(fault_texts):
        encoded = tokenizer(sentence, padding="max_length", truncation=True,
                            max_length=max_length, return_tensors="pt").to(device)
        
        output = model(**encoded)  # output.last_hidden_state: [1, 128, 384]
        token_embeds = output.last_hidden_state.squeeze(0).cpu()  # [128, 384]
        
        token_level_embeddings.append(token_embeds)

# Save as a tensor [N, 128, 384]
token_level_embeddings = torch.stack(token_level_embeddings)
torch.save(token_level_embeddings, "minilm_token_embeddings.pt")



In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load fault logs
df = pd.read_excel("fault-logs-o-ran-350.xls")
#df = pd.read_excel("proj-training-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Tokenizer and GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Dataset
class FaultDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.examples = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    def __len__(self):
        return len(self.examples["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.examples.items()}

dataset = FaultDataset(fault_texts, tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-fault",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("gpt2-finetuned-fault")
tokenizer.save_pretrained("gpt2-finetuned-fault")


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load MiniLM token embeddings
minilm_token_embeds = torch.load("minilm_token_embeddings.pt")

# Load texts
df = pd.read_excel("fault-logs-o-ran-350.xls")
fault_texts = df["Fault Description"].astype(str).tolist()

# Load GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault").to(device)
gpt2.eval()

for p in gpt2.parameters():
    p.requires_grad = False

MAX_LEN = 128

class ProjectorDataset(Dataset):

    def __init__(self, embeds, texts):

        tok = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        self.embeds = embeds
        self.input_ids = tok["input_ids"]
        self.attn_mask = tok["attention_mask"]

        # ignore pad tokens
        self.labels = self.input_ids.clone()
        self.labels[self.attn_mask == 0] = -100

    def __len__(self):
        return len(self.embeds)

    def __getitem__(self, idx):
        return {
            "embed": self.embeds[idx],
            "labels": self.labels[idx],
            "mask": self.attn_mask[idx]
        }

dataset = ProjectorDataset(minilm_token_embeds, fault_texts)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# projector
class EmbeddingProjector(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(384,512),
            nn.GELU(),
            nn.Linear(512,768),
            nn.GELU(),
            nn.Linear(768,768)
        )

    def forward(self,x):

        B,T,D = x.size()

        x = x.view(B*T,D)
        x = self.net(x)
        x = x.view(B,T,768)

        return x


projector = EmbeddingProjector().to(device)

optimizer = torch.optim.Adam(projector.parameters(), lr=1e-4)

EPOCHS = 300

for epoch in range(EPOCHS):

    projector.train()
    total_loss = 0

    for batch in loader:

        embeds = batch["embed"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        proj = projector(embeds)

        outputs = gpt2(
            inputs_embeds=proj,
            attention_mask=mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss {total_loss/len(loader):.4f}")

torch.save(projector.state_dict(),"trained_projector-new.pt")

In [ ]:
import torch
import csv
from torch import nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Generator architecture
# -----------------------------
class Generator(nn.Module):

    def __init__(self, latent_dim=100, output_dim=384):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,output_dim)
        )

    def forward(self,z):
        return self.model(z)


# -----------------------------
# Load trained generators
# -----------------------------
K = 30
generators = []

for j in range(1,K+1):

    G = Generator().to(device)

    G.load_state_dict(
        torch.load(f"trained_generator_{j}.pt",map_location=device)
    )

    G.eval()
    generators.append(G)

print(f"Loaded {K} generators")


# -----------------------------
# Load GPT-2
# -----------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault")
gpt2 = gpt2.to(device)
gpt2.eval()

print("GPT-2 loaded")


# -----------------------------
# Projector architecture
# (must match training)
# -----------------------------
class EmbeddingProjector(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(384,512),
            nn.GELU(),
            nn.Linear(512,768),
            nn.GELU(),
            nn.Linear(768,768)
        )

    def forward(self,x):

        B,T,D = x.size()

        x = x.view(B*T,D)
        x = self.net(x)
        x = x.view(B,T,768)

        return x


projector = EmbeddingProjector().to(device)

projector.load_state_dict(
    torch.load("trained_projector-new.pt",map_location=device)
)

projector.eval()

print("Projector loaded")


# -----------------------------
# Generation settings
# -----------------------------
latent_dim = 100
num_samples_per_gen = 500

prefix_len = 32

decoded_outputs = []

total_expected = K * num_samples_per_gen
sample_counter = 0

print(f"\nExpected samples: {total_expected}\n")


# -----------------------------
# Generate samples
# -----------------------------
with torch.no_grad():

    for g_idx,G in enumerate(generators):

        print(f"\n🔁 Generator {g_idx+1}/{K}")

        z = torch.randn(num_samples_per_gen,latent_dim).to(device)

        fake_emb = G(z)             # [N,384]

        # Convert sentence embedding → token embeddings
        fake_emb = fake_emb.unsqueeze(1).repeat(1,prefix_len,1)

        # Add small noise
        fake_emb = fake_emb + 0.01 * torch.randn_like(fake_emb)

        projected = projector(fake_emb)  # [N,prefix_len,768]

        for i in range(num_samples_per_gen):

            prefix = projected[i].unsqueeze(0)  # [1,prefix_len,768]

            output_ids = gpt2.generate(

                inputs_embeds=prefix,

                max_new_tokens=80,

                do_sample=True,

                temperature=0.8,

                top_k=40,

                top_p=0.9,

                repetition_penalty=1.15,

                pad_token_id=tokenizer.eos_token_id
            )

            decoded = tokenizer.decode(
                output_ids[0],
                skip_special_tokens=True
            )

            decoded_outputs.append(
                (f"G{g_idx+1}",f"Sample {i+1}",decoded)
            )

            sample_counter += 1

            # Show progress every 100 samples
            if sample_counter % 100 == 0:
                print(f"Progress: {sample_counter} / {total_expected}")


# -----------------------------
# Remove empty generations
# -----------------------------
decoded_outputs = [
    row for row in decoded_outputs
    if len(row[2].strip()) > 5
]


# -----------------------------
# Save results
# -----------------------------
output_file = "synthetic_fault_logs_final-2026-15k.csv"

with open(output_file,"w",newline="",encoding="utf-8") as f:

    writer = csv.writer(f)

    writer.writerow([
        "Generator",
        "Sample Number",
        "Generated Fault Description"
    ])

    for row in decoded_outputs:
        writer.writerow(row)


print(f"\nSaved {len(decoded_outputs)} samples to {output_file}")

In [ ]:
# fine_tune_gpt2.py
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Load fault logs
df = pd.read_excel("fault-logs-o-ran-12k.xlsx", engine="openpyxl")
fault_texts = df["Fault Description"].astype(str).tolist()

# Tokenizer and GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Dataset
class FaultDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.examples = tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    def __len__(self):
        return len(self.examples["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.examples.items()}

dataset = FaultDataset(fault_texts, tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-fault-12k",
    per_device_train_batch_size=4,
    num_train_epochs=8,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("gpt2-finetuned-fault-12k")
tokenizer.save_pretrained("gpt2-finetuned-fault-12k")


In [ ]:
# generate_faults_from_gan.py
import torch
import csv
from torch import nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# --- Generator class ---
class Generator(nn.Module):
    def __init__(self, latent_dim=100, output_dim=384):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    def forward(self, z):
        return self.model(z)

# --- Projector class (must match training structure) ---
class EmbeddingProjector(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(384, 768)
    def forward(self, x):
        return self.linear(x)

# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 100
K = 20
num_samples_per_gen = 60

# --- Load all trained generators ---
generators = []
for j in range(1, K + 1):
    G = Generator().to(device)
    G.load_state_dict(torch.load(f"trained_generator_{j}.pt", map_location=device))
    G.eval()
    generators.append(G)

# --- Load fine-tuned GPT-2 ---
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-finetuned-fault")
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2-finetuned-fault").to(device)
gpt2.eval()

# --- Load trained projector ---
projector = EmbeddingProjector().to(device)
projector.load_state_dict(torch.load("trained_projector-old.pt", map_location=device))
projector.eval()

# --- Generate and decode ---
decoded_outputs = []

with torch.no_grad():
    for g_idx, G in enumerate(generators):
        print(f"Generating from Generator {g_idx + 1}...")
        z = torch.randn(num_samples_per_gen, latent_dim).to(device)
        fake_emb = G(z)                          # [N, 384]
        projected = projector(fake_emb)          # [N, 768]

        for i, emb in enumerate(projected):
            input_emb = emb.unsqueeze(0).unsqueeze(1)  # [1, 1, 768]
            output_ids = gpt2.generate(
                inputs_embeds=input_emb,
                max_length=128,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
            decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            decoded_outputs.append((f"G{g_idx + 1}", f"Sample {i + 1}", decoded))
            print(f"✓ G{g_idx + 1}, Sample {i + 1}: {decoded[:60]}...")

# --- Save to CSV ---
output_file = "inference-data-new.csv"
with open(output_file, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Generator", "Sample Number", "Generated Fault Description"])
    for row in decoded_outputs:
        writer.writerow(row)

print(f"\n✅ Saved {len(decoded_outputs)} samples to {output_file}")


In [ ]:
import pandas as pd

# Group outputs by generator index
gen_ids = []
texts = []

for g_idx in range(K):
    for i in range(num_samples_per_gen):
        idx = g_idx * num_samples_per_gen + i
        gen_ids.append(f"G{g_idx + 1}")
        texts.append(decoded_outputs[idx])

# Create DataFrame with generator ID
df = pd.DataFrame({
    "Generator": gen_ids,
    "Synthetic Fault Log": texts
})

# Save to CSV
df.to_csv("synthetic_fault_logs_by_generator-12k.csv", index=False)
print("✅ Saved to synthetic_fault_logs_by_generator-12k.csv")



In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

# Load data with generator column
df_fake = pd.read_csv("gmm_inference_evaluation.csv")
fake_logs = df_fake["Synthetic Fault Log"].dropna().astype(str).tolist()
generator_ids = df_fake["Generator"].tolist()

# Load real logs
df_real = pd.read_excel("fault-logs-o-ran-350.xls")
real_logs = df_real["Fault Description"].dropna().astype(str).tolist()

# Models
embedder = SentenceTransformer("all-MiniLM-L6-v2")
rouge = Rouge()
smoothie = SmoothingFunction().method1

# Evaluation records
records = []

for i, fake in enumerate(fake_logs):
    fake_emb = embedder.encode([fake])
    real_embs = embedder.encode(real_logs)
    sims = cosine_similarity(fake_emb, real_embs)[0]
    best_idx = sims.argmax()
    ref = real_logs[best_idx]

    # BLEU & ROUGE
    bleu = sentence_bleu([ref.split()], fake.split(), smoothing_function=smoothie)
    rouge_score = rouge.get_scores(fake, ref)[0]['rouge-l']['f']
    cosine_sim = sims[best_idx]

    records.append({
        "Generator": generator_ids[i],
        "Fake Log": fake,
        "Best Match Real Log": ref,
        "BLEU": bleu,
        "ROUGE_L_F1": rouge_score,
        "Cosine Similarity": cosine_sim
    })

# Save detailed evaluation
results_df = pd.DataFrame(records)
results_df.to_csv("synthetic_fault_log_eval_detailed.csv", index=False)

# Summary metrics per generator
summary_df = results_df.groupby("Generator")[["BLEU", "ROUGE_L_F1", "Cosine Similarity"]].mean()
summary_df.reset_index(inplace=True)
summary_df.to_csv("evaluation_metrics_by_generator.csv", index=False)

print("✅ Evaluation complete. Metrics saved to:")
print("- synthetic_fault_log_eval_detailed.csv")
print("- evaluation_metrics_by_generator.csv")
